# Week 8 — Logistic regression: controlled comparison

### Deliverable
- Baseline vs extended model + odds ratios + interpretation


In [ ]:
# Core imports (kept minimal for beginners)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Dataset URL (UCI Heart Disease - Cleveland)
UCI_URL = "https://www.archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names for processed.cleveland.data (14 columns commonly used in teaching)
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","num"]


In [ ]:
import statsmodels.api as sm


In [ ]:
import ssl
import io
import urllib.request # Added this import

def load_raw():
    # Create an unverified SSL context to bypass certificate verification
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Open the URL with the unverified context
    with urllib.request.urlopen(UCI_URL, context=ctx) as url_response:
        # Read the content and decode it
        s = url_response.read().decode('utf-8')

    # Use io.StringIO to make the string behave like a file for pandas.read_csv
    df_raw = pd.read_csv(io.StringIO(s), header=None, names=COLS)
    return df_raw

def coerce_types(df_raw):
    # Missing values are sometimes encoded as "?"
    df = df_raw.replace("?", np.nan).copy()

    # Convert numeric-looking columns
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Binary target: disease present if num > 0 (UCI convention)
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_raw = load_raw()
df = coerce_types(df_raw)

df.head()


In [ ]:
def clean_heart(df_raw):
    df = df_raw.replace("?", np.nan).copy()
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["disease"] = (df["num"] > 0).astype(int)
    return df

df_clean = clean_heart(df_raw)

base_features = ["age","sex"]
extra_features = ["thalach","oldpeak","exang"]  # TODO
features = base_features + extra_features

df_model = df_clean.dropna(subset=features + ["disease"]).copy()
df_model.shape


## Baseline model


In [ ]:
X_base = sm.add_constant(df_model[base_features])
y = df_model["disease"]
m_base = sm.Logit(y, X_base).fit(disp=False)
print(m_base.summary())


## Extended model


In [ ]:
X_ext = sm.add_constant(df_model[features])
m_ext = sm.Logit(y, X_ext).fit(disp=False)
print(m_ext.summary())


## Odds ratios + 95% CI


In [ ]:
def odds_ratios_with_ci(model):
    params = model.params
    se = model.bse
    or_ = np.exp(params)
    lo = np.exp(params - 1.96*se)
    hi = np.exp(params + 1.96*se)
    return pd.DataFrame({"odds_ratio": or_, "ci_low": lo, "ci_high": hi, "p_value": model.pvalues})

display(odds_ratios_with_ci(m_base))
display(odds_ratios_with_ci(m_ext))


## TODO — Interpret two coefficients
- Variable 1:
  - Interpretation:
  - What it does NOT prove:

- Variable 2:
  - Interpretation:
  - What it does NOT prove:

Unmeasured confounder you worry about:


 Variable 1: thalach (maximum heart rate achieved)
  · Interpretation: Each additional beat per minute in maximum heart rate is associated with a 3% decrease in the odds of heart disease (OR ≈ 0.97), holding all other variables constant.
  · What it does NOT prove: Higher heart rate causes lower disease risk. It could be that healthier hearts are able to beat faster under stress, or that unmeasured fitness levels drive both.
  
· Variable 2: exang (exercise induced angina, 1 = yes)
  · Interpretation: Having exercise-induced angina is associated with 3 times higher odds of heart disease (OR ≈ 3.00), adjusted for age, sex, thalach, and oldpeak.
  · What it does NOT prove: Angina causes heart disease. It is a symptom, not a cause. The association may reflect underlying coronary blockages.

· Unmeasured confounder you worry about:
  Physical fitness / exercise capacity – not measured in this dataset. Fitness affects both thalach (fit people have higher max heart rates) and exang (fit people are less likely to have angina), and also directly reduces disease risk. Without controlling for fitness, the effects of thalach and exang are likely biased.